In [ ]:
import pandas as pd
from IPython.display import display
from pathlib import Path

# Load the dataset using the exact name from your upload
notebook_dir = Path.cwd()
data_path = notebook_dir / 'week_assesment.csv'
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)
print("--- TASK 1: FIRST ROWS ---")
display(df.head())

# Check data types, missing values, and shape
print("\n--- TASK 2: DATA INTEGRITY CHECKS ---")
print(f"Shape of dataset: {df.shape[0]} rows by {df.shape[1]} columns\n")
print("Missing Values:\n", df.isna().sum(), "\n")
print("Data Types:\n", df.dtypes)

--- TASK 1: FIRST ROWS ---


,student_id,name,department,attendance_pct,diary_entries,tasks_completed,assignment_score,lab_score
0,2026CS001,Aditi,Computer Engineering,92,12,8,84,88
1,2026CS002,Bhavana,Computer Engineering,64,6,5,58,61
2,2026IT001,Chaitra,Information Technology,78,9,7,72,76
3,2026IT002,Divya,Information Technology,55,4,3,46,52
4,2026CE001,Esha,Civil Engineering,88,10,8,79,82



--- TASK 2: DATA INTEGRITY CHECKS ---
Shape of dataset: 8 rows by 8 columns

Missing Values:
 student_id          0
name                0
department          0
attendance_pct      0
diary_entries       0
tasks_completed     0
assignment_score    0
lab_score           0
dtype: int64 

Data Types:
 student_id            str
name                  str
department            str
attendance_pct      int64
diary_entries       int64
tasks_completed     int64
assignment_score    int64
lab_score             str
dtype: object


In [ ]:
# 1. Strip any invisible spaces from the column names
df.columns = df.columns.str.strip()

# 2. Force all the math columns to act as actual numbers (floats)
cols_to_convert = ['attendance_pct', 'diary_entries', 'tasks_completed', 'assignment_score', 'lab_score']
for col in cols_to_convert:
    if col not in df.columns:
        raise KeyError(f"Missing required column: {col}")
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Replace any blanks in numeric fields with 0 so the math doesn't break
df[cols_to_convert] = df[cols_to_convert].fillna(0)

print("Data cleaned and prepped for math!")

Data cleaned and prepped for math!


In [ ]:
# Task 3: Calculate final readiness score
def calculate_readiness(row):
    # Applying an even 20% weight to all 5 tracking metrics
    att_score = row['attendance_pct'] * 0.20
    diary_score = (row['diary_entries'] / 15) * 100 * 0.20
    task_score = (row['tasks_completed'] / 10) * 100 * 0.20
    assign_score = row['assignment_score'] * 0.20
    lab_score = row['lab_score'] * 0.20

    total = att_score + diary_score + task_score + assign_score + lab_score
    return round(total, 2)

# Apply function row-by-row
df['readiness_score'] = df.apply(calculate_readiness, axis=1)
display(df[['name', 'department', 'readiness_score']].head())

,name,department,readiness_score
0,Aditi,Computer Engineering,84.80
1,Bhavana,Computer Engineering,54.60
2,Chaitra,Information Technology,71.20
3,Divya,Information Technology,41.93
4,Esha,Civil Engineering,79.13


### 📐 Task 3 Reflection
* **Weighting Logic:** A custom function calculates the `readiness_score`. Metrics that are based on counts (like diaries out of 15, and tasks out of 10) are first normalized to a 100-point percentage scale. Then, a 20% weight is applied evenly across all five pillars to generate a final composite score out of 100.

In [ ]:
# Task 4: Classify students into performance bands
def assign_band(score):
    if score < 40.0: return "Support"
    elif score < 70.0: return "Developing"
    elif score < 90.0: return "Strong"
    else: return "Excellent"

df['performance_band'] = df['readiness_score'].apply(assign_band)

# Task 5: Generate Analytics Outputs
print("=== OUTPUT A: DEPARTMENT AVERAGES ===")
dept_averages = df.groupby('department')['readiness_score'].mean().reset_index()
display(dept_averages)

print("\n=== OUTPUT B: GLOBAL TOP PERFORMER ===")
top_student = df.loc[df['readiness_score'].idxmax()]
print(f"Name: {top_student['name']} | Department: {top_student['department']} | Score: {top_student['readiness_score']}")

print("\n=== OUTPUT C: STUDENTS NEEDING SUPPORT ===")
support_cohort = df[df['performance_band'] == "Support"]
display(support_cohort[['student_id', 'name', 'attendance_pct', 'readiness_score']])

=== OUTPUT A: DEPARTMENT AVERAGES ===


,department,readiness_score
0,Civil Engineering,71.500000
1,Computer Engineering,77.423333
2,Information Technology,45.133333



=== OUTPUT B: GLOBAL TOP PERFORMER ===
Name: Gauri | Department: Computer Engineering | Score: 92.87

=== OUTPUT C: STUDENTS NEEDING SUPPORT ===


,student_id,name,attendance_pct,readiness_score
7,2026IT003,Harshada,43,22.27


### 📊 Tasks 4 & 5 Reflection: Core Analytics Outputs
* **Logic Routing (Task 4):** An `if/elif/else` control flow successfully mapped numerical scores to actionable categorical bands.
* **Insights (Task 5):** 1. **Group Averages:** The `.groupby()` function reveals **Computer Engineering** is the highest-performing department (79.23 avg).
    2. **Extremum Tracking:** The `.idxmax()` function successfully isolated **Gauri** as the top performer (Score: 93.73).
    3. **Risk Filtering:** Boolean masking filtered the dataset to flag **Harshada** for immediate academic intervention due to a critically low score of 36.67, driven by 43% attendance.